In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
import os
import sys
import argparse

def count_beacons_per_room(columns):
    beacon_columns = [col for col in columns if '-' in col]
    room_numbers = [col.split('-')[0] for col in beacon_columns]
    room_counts = Counter(room_numbers)
    result = pd.DataFrame({
        'Room': list(room_counts.keys()),
        'Beacon Count': list(room_counts.values())
    })
    result['Room'] = result['Room'].astype(int)
    result = result.sort_values('Room').reset_index(drop=True)
    return result

def plot_artworks_and_beacons_combined(df, output_path):
    sns.set_style('whitegrid')
    plt.rcParams['font.family'] = 'sans-serif'
    fig, ax = plt.subplots(figsize=(10, 5))
    room_counts = df.groupby('room')['artwork'].nunique().reset_index()
    room_counts.columns = ['Room', 'Artworks']
    beacon_data = {
        'Room': [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28],
        'Beacons': [7, 8, 4, 6, 5, 6, 1, 3, 6, 3, 5, 11, 13]
    }
    beacon_df = pd.DataFrame(beacon_data)
    room_counts['Room'] = room_counts['Room'].astype(str)
    beacon_df['Room'] = beacon_df['Room'].astype(str)
    all_rooms = sorted(list(set(room_counts['Room'].tolist() + beacon_df['Room'].tolist())), key=lambda x: int(x))
    complete_df = pd.DataFrame({'Room': all_rooms})
    complete_df = complete_df.merge(room_counts, on='Room', how='left').fillna(0)
    complete_df = complete_df.merge(beacon_df, on='Room', how='left').fillna(0)
    plot_data = pd.melt(complete_df, id_vars=['Room'], value_vars=['Artworks', 'Beacons'], var_name='Type', value_name='Count')
    bar_plot = sns.barplot(x='Room', y='Count', hue='Type', data=plot_data, palette=['steelblue', 'indianred'], ax=ax)
    for p in bar_plot.patches:
        if p.get_height() > 0:
            bar_plot.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=10)
    plt.rcParams.update({'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 15, 'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 14})
    bar_plot.legend(title='')
    sns.despine()
    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Grafico Artworks/Beacons salvato in: {output_path}")
    plt.close(fig)

def plot_rssi_comparison(df_android, df_ios, artwork_id, beacons_to_compare, output_path):
    subset_android = df_android[df_android.artwork == artwork_id][beacons_to_compare].replace(110, float('nan'))
    subset_ios = df_ios[df_ios.artwork == artwork_id][beacons_to_compare].replace(110, float('nan'))
    subset_android['platform'] = 'Android'
    subset_ios['platform'] = 'iOS'
    combined_data = pd.concat([subset_android, subset_ios])
    combined_data_long = pd.melt(combined_data, id_vars=['platform'], value_vars=beacons_to_compare, var_name='Signal', value_name='RSSI').dropna()
    sns.set_style('whitegrid')
    plt.rcParams['font.family'] = 'sans-serif'
    fig, ax = plt.subplots(figsize=(8, 6))
    box_plot = sns.boxplot(x='Signal', y='RSSI', hue='platform', data=combined_data_long, palette=['steelblue', '#e74c3c'], ax=ax)
    ax.set_xlabel('Beacon', fontsize=12)
    ax.set_ylabel('RSS Value (dBm)', fontsize=12)
    ax.legend(title='', frameon=True)
    sns.despine()
    plt.rcParams.update({'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 15, 'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 14})
    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Grafico Confronto RSSI salvato in: {output_path}")
    plt.close(fig)

parser = argparse.ArgumentParser(description="Generate figures for the BAR dataset paper.")
parser.add_argument('--raw-data-dir', type=str, default='/content/data/raw', help='Directory containing raw JSON data.')
parser.add_argument('--fingerprint-dir', type=str, default='/content/data/fingerprints', help='Directory containing fingerprint CSV files.')
parser.add_argument('--output-dir', type=str, default='/content/drive/My Drive/Tesi Magistrale/Grafici_Analisi/', help='Directory to save generated figures.')

args, unknown = parser.parse_known_args()

os.makedirs(args.output_dir, exist_ok=True)

print("Caricamento dati dai fingerprint...")
try:
    a1_path = os.path.join(args.fingerprint_dir, "android_session_1_1.csv")
    a1 = pd.read_csv(a1_path)

    i1_path = os.path.join(args.fingerprint_dir, "ios_session_1_3.csv")
    i1 = pd.read_csv(i1_path)

    # a2_path = os.path.join(args.fingerprint_dir, "android_session_2_2.csv")
    # a2 = pd.read_csv(a2_path)
    # i2_path = os.path.join(args.fingerprint_dir, "ios_session_2_4.csv")
    # i2 = pd.read_csv(i2_path)

except FileNotFoundError as e:
    print(f"Errore: File CSV non trovato - {e}. Assicurati di aver decompresso all.zip.")
    sys.exit(1)

print("Dati caricati con successo.")

output_path_artworks = os.path.join(args.output_dir, 'artworks_and_beacons_combined.png')
plot_artworks_and_beacons_combined(a1, output_path_artworks)

beacons_for_rssi_plot = ["26-1", "26-2", "26-3"]
artwork_for_rssi_plot = 19
output_path_rssi = os.path.join(args.output_dir, 'platform_rssi_comparison.png')
plot_rssi_comparison(a1, i1, artwork_for_rssi_plot, beacons_for_rssi_plot, output_path_rssi)

print("\nGenerazione grafici completata!")